In [ ]:
from pathlib import Path
from IPython.display import Image, SVG, display
import getpass
import os
import re
import time

# Works when this notebook starts in notebooks/, the repository root, or Gadi.
candidates = [Path.cwd(), *Path.cwd().parents,
              Path(os.path.expandvars("/scratch/vp91/$USER/intro-to-cuda"))]
repo_root = next((path for path in candidates
                  if (path / "exercises/CMakeLists.txt").is_file()
                  and (path / "job-script").is_dir()), None)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from the intro-to-cuda checkout.")
os.chdir(repo_root)
job_dir = repo_root / "job-script"
username = getpass.getuser()


def show_job_output(job_name, job_id):
    job_number = job_id.split(".")[0]
    log = job_dir / f"{job_name}.o{job_number}"
    if log.is_file():
        print(log.read_text())
    else:
        print(f"Output is not available yet: {log.name}. Check qstat, then rerun this cell.")


def outputs_ready(paths, submitted_at):
    if all(path.is_file() and path.stat().st_mtime >= submitted_at for path in paths):
        return True
    print("New results are not available yet. Wait for this job to finish, then rerun this cell.")
    return False

print(f"Repository: {repo_root}")

# CUDA Events


> **Overview**
>
> 
>  * **Time:** 30 min
> 
>  1. Learn about CUDA events and their usage.
>  1. Understand how to measure elapsed time using CUDA events.

CUDA events are a powerful feature that allows you to measure the time taken by operations on the GPU.
They can be used to synchronize between different streams and to profile the performance of your CUDA
applications. Events are lightweight and provide a way to track the completion of operations without
blocking the CPU.

## CUDA Event Basics

In [ ]:
# CUDA Events
display(Image(filename=str(repo_root / "docs/source/tutorial/figs/cuda-events.png")))

CUDA events are used to mark points in time in your CUDA application. They can be created, recorded, and
queried to determine when certain operations have completed. Events can be used to measure the elapsed time
between two points in your code, which is useful for performance profiling. CUDA events are created using
the `cudaEventCreate` function, and you can record an event using `cudaEventRecord`.

```cpp
cudaEvent_t start, stop;
cudaEventCreate(&start);
cudaEventCreate(&stop);

// Record the start event
cudaEventRecord(start, 0);

// Perform some GPU operations here

// Record the stop event
cudaEventRecord(stop, 0);

// Wait for the stop event to complete
cudaEventSynchronize(stop);

float milliseconds = 0;
cudaEventElapsedTime(&milliseconds, start, stop);
printf("Elapsed time: %f ms\n", milliseconds);
```

### Timing work with events on the default stream

The host queues the start marker, the GPU operations, and the stop marker in
stream 0. The GPU records each timestamp when it reaches that marker. Earlier
work in the same stream can delay the start marker and falls outside the
measured interval.

The host waits in `cudaEventSynchronize(stop)` until the stop marker completes.
Then `cudaEventElapsedTime` returns the difference between the two GPU
timestamps in milliseconds. The animation uses illustrative timing and runs
without CUDA or PBS.


In [ ]:
from IPython.display import HTML, display

# Positions are illustrative. Both lanes share the same horizontal time axis.
left, start, stop, right = 100, 260, 540, 700
lanes = [
    ("Host", 47, [
        (100, 164, "Queue start", "#1d4ed8"),
        (164, 229, "Queue work", "#6d28d9"),
        (229, 294, "Queue stop", "#047857"),
        (294, stop, "Wait in cudaEventSynchronize(stop)", "#334155"),
        (stop, right, "Read elapsed time", "#1d4ed8"),
    ]),
    ("GPU · stream 0", 107, [
        (left, start, "Earlier queued work", "#475569"),
        (start, stop, "Timed GPU operations", "#6d28d9"),
    ]),
]
parts = []
for lane, y, operations in lanes:
    parts.append(f'<text x="12" y="{y + 21}" font-weight="700">{lane}</text>')
    for begin, end, label, color in operations:
        parts.append(
            f'<rect x="{begin + 1}" y="{y}" width="{end - begin - 2}" height="32" rx="4" fill="{color}"/>'
            f'<text x="{(begin + end) / 2}" y="{y + 20}" text-anchor="middle" fill="white">{label}</text>'
        )
for x, label, color in [(start, "start timestamp", "#1d4ed8"), (stop, "stop timestamp", "#047857")]:
    parts.append(
        f'<path d="M{x} 101 V160" stroke="{color}" stroke-width="2"/>'
        f'<circle cx="{x}" cy="123" r="4" fill="{color}" stroke="white"/>'
        f'<text x="{x}" y="174" text-anchor="middle" fill="{color}">{label}</text>'
    )
parts.append(
    f'<path d="M{start} 184 v8 H{stop} v-8" fill="none" stroke="#475569"/>'
    f'<text x="{(start + stop) / 2}" y="210" text-anchor="middle">Elapsed time = stop − start</text>'
)

display(HTML("""
<style>
.cuda-event-timing { max-width: 720px; }
.cuda-event-timing svg { display: block; width: 100%; height: auto; }
.cuda-event-timing .event-cursor { animation: cuda-event-sweep 9s linear infinite; }
.cuda-event-timing input:checked ~ svg .event-cursor { animation-play-state: paused; }
@keyframes cuda-event-sweep { to { transform: translateX(600px); } }
@media (prefers-reduced-motion: reduce) {
    .cuda-event-timing .event-cursor { display: none; }
}
</style>
<div class="cuda-event-timing">
<input type="checkbox" id="cuda-event-timing-pause">
<label for="cuda-event-timing-pause">Pause cursor</label>
<svg viewBox="0 0 720 230" role="img" aria-labelledby="cuda-event-timing-title">
  <title id="cuda-event-timing-title">CUDA events time work on the default stream</title>
  <rect width="720" height="230" rx="8" fill="#f8fafc"/>
  <g font-family="system-ui, sans-serif" font-size="11" fill="#172033">
    <text x="100" y="28" font-size="15" font-weight="700">Event timing on the default stream</text>
""" + "".join(parts) + """
  </g>
  <g class="event-cursor" aria-hidden="true">
    <path d="M100 39 V156" stroke="white" stroke-width="4"/>
    <path d="M100 39 V156" stroke="#111827" stroke-width="2" stroke-dasharray="4 3"/>
  </g>
</svg>
</div>
"""))


## Cross stream Synchronization


CUDA events can also be used for cross-stream synchronization. When you record an event in one stream,
it can be waited on in another stream. This allows you to synchronize operations across different streams without blocking the CPU.

```cpp
cudaStream_t stream1, stream2;
cudaStreamCreate(&stream1);
cudaStreamCreate(&stream2);

cudaEvent_t event;
cudaEventCreate(&event);

// Perform the producer operation in stream1
kernel1<<<blocks, threads, 0, stream1>>>(...);

// Capture completion of the producer, then make stream2 wait for it
cudaEventRecord(event, stream1);

// Wait for the event in stream2
cudaStreamWaitEvent(stream2, event, 0);

// Perform some operations in stream2 that depend on the completion of stream1
kernel2<<<blocks, threads, 0, stream2>>>(...);

// Wait before releasing resources used by the consumer
cudaStreamSynchronize(stream2);

// Cleanup
cudaEventDestroy(event);
cudaStreamDestroy(stream1);
cudaStreamDestroy(stream2);
```

> **Explanation**
>
> 
>  `cudaStreamWaitEvent` is used to make a stream wait for an event recorded in another stream.

## Timing events across streams


When timing events across streams, you can record events in one stream and then wait for those events
in another stream. This allows you to measure the time taken by operations in different streams without
blocking the CPU.

```cpp
cudaStream_t stream1, stream2;
cudaStreamCreate(&stream1);
cudaStreamCreate(&stream2);

cudaEvent_t start, stop, producerDone;
cudaEventCreate(&start);
cudaEventCreate(&stop);
cudaEventCreate(&producerDone);

// Record the start event in stream1
cudaEventRecord(start, stream1);

// Perform some operations in stream1
kernel1<<<blocks, threads, 0, stream1>>>(...);

// Make the stop marker depend on completion of kernel1
cudaEventRecord(producerDone, stream1);
cudaStreamWaitEvent(stream2, producerDone, 0);
cudaEventRecord(stop, stream2);

// This host-side wait ensures both timestamps are available
cudaEventSynchronize(stop);

float milliseconds = 0;
cudaEventElapsedTime(&milliseconds, start, stop);
printf("Elapsed time across streams: %f ms\n", milliseconds);

cudaEventDestroy(start);
cudaEventDestroy(stop);
cudaEventDestroy(producerDone);
cudaStreamDestroy(stream1);
cudaStreamDestroy(stream2);
```

> **Important**
>
> 1. You cannot use an event recorded in one stream to measure the execution time of operations that occur in another stream unless you enforce proper synchronization.
> 1. If streams are non-blocking and concurrent, incorrect usage may lead to race conditions or invalid timings.

> **Key Points**
>
> 
>  1. CUDA events are used to measure elapsed time and synchronize operations across streams.
>  1. Events can be created, recorded, and queried to determine the completion of operations.
>  1. Cross-stream synchronization is achieved using `cudaStreamWaitEvent`.
>  1. Timing events across streams requires careful synchronization to ensure accurate measurements.
>  1. Events are lightweight and do not block the CPU, making them suitable for performance profiling.

## Event ordering in these examples

The notebook corrects two ordering issues in the RST examples: the cross-stream dependency is recorded **after** the producer kernel, and the cross-stream stop event waits for producer completion. An event captures work already submitted to its stream; later submissions are not included. See [NVIDIA event documentation](https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__EVENT.html).

`cudaStreamWaitEvent` queues a device-side dependency; `cudaEventSynchronize` waits on the CPU. Recording events can be asynchronous, but explicit host synchronization blocks until completion.

The snippets above illustrate API ordering and contain omitted kernels and launch parameters (`...`), so they are not standalone programs. The executable example below is complete.

## Exercise 08: CUDA Events

**Time: 20 min**

Inspect the event markers in the supplied program, then submit its documented job.
[Exercise instructions](../docs/source/tutorial/exercises/08-events.rst).

Run the setup cell first. Build the exercises once in
[02-gpu.ipynb](02-gpu.ipynb) before submitting jobs from this notebook.
After editing CUDA source, return to that notebook to rebuild.

Submission cells use the same `qsub` commands as the exercise documentation,
from `job-script`, and immediately show `qstat`. A submitted job may still be queued.
Use the monitoring cell to check again; rerunning a submission cell creates another job.
Read the PBS output and inspect generated files after the job finishes.

<a href="../exercises/08-events/vector-add-async.cu" target="_blank" rel="noopener noreferrer">Open vector-add-async.cu</a>

In [ ]:
# Submit a new job, then inspect the queue.
submitted_at = time.time()
submission = !cd "{job_dir}" && qsub 08-events.pbs
print("\n".join(submission))
!qstat -u "{username}"

# Keep the returned job ID so later cells read this run's PBS log.
job_id = next((line.strip() for line in submission
                 if re.fullmatch(r"\d+(?:\.[A-Za-z0-9_.-]+)?", line.strip())), None)
if job_id is None:
    raise RuntimeError("qsub did not return a job ID. Check its output above.")


### Check the queue again

Run this cell to monitor jobs without submitting another one. Once your job is no
longer listed, rerun its output/visualisation cells. Inspect the PBS log for errors.

In [ ]:
!qstat -u "{username}"


In [ ]:
!qstat -u "{username}"
show_job_output("08-events", job_id)


### Interpret the timings

Check result verification reports `PASSED`. Each iteration records its start
before host-to-device copies and its stop after the device-to-host copy in the
same stream. The interval covers transfers and kernel work, not just the kernel.
Do all iterations take the same time? Repeat the job and compare variability.